# validacion final

tests estadisticos y benchmark vs baselines

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from omnievo import (
    DataGenerator,
    GeneticOptimizer,
    cross_validate,
    run_benchmark,
    compare_baselines,
    plot_convergence,
    plot_weights,
    plot_comparison,
)

In [ ]:
gen = DataGenerator(n_users=2000, random_state=42)
df = gen.generate()
channels = gen.get_channel_names()

X = df[channels].values
y = df['LTV_real'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

## entrenar con mejores params

In [ ]:
# params del notebook anterior (o ajustar)
params = {
    "population_size": 50,
    "generations": 100,
    "cxpb": 0.6,
    "mutpb": 0.2,
}

opt = GeneticOptimizer(**params, random_state=42)
result = opt.fit(X_train, y_train)

## cv para los tests

In [ ]:
cv = cross_validate(X, y, k_folds=5, optimizer_params=params)
print(f"rmse: {cv['rmse_mean']:.4f} +/- {cv['rmse_std']:.4f}")

## benchmark

In [ ]:
report = run_benchmark(
    X_train, y_train, X_test, y_test,
    ga_weights=result['best_weights'],
    channel_names=channels,
    cv_results=cv,
)

## graficas

In [ ]:
plot_convergence(result);

In [ ]:
plot_weights(result['best_weights'], channels);

In [ ]:
plot_comparison(report['comparison_df']);

## peso por tipo de canal

In [ ]:
types = gen.get_channel_types()
w = result['best_weights']

for t, chs in types.items():
    idx = [channels.index(c) for c in chs]
    total = sum(w[i] for i in idx)
    print(f"{t}: {total*100:.1f}%")

## resultado

si el t-test sale significativo, podemos decir que el GA es mejor que el baseline uniforme. en general los canales IoT tienen mas peso porque estan mas correlacionados con el gasto real.

In [ ]:
print(f"mejora vs uniforme: {report['improvement_pct']:.1f}%")
print(f"h0 rechazada: {report['hypothesis_rejected']}")